# Outcome-only concept model with robust token-level late interaction

This version does **not** use note-level concept annotations for training. It uses concise ICD-10 names and synonyms, contextualized alias prototypes, local-window phrase coverage, and a fixed exact-alias branch. When a unique stored alias occurs verbatim in the tokenized note, its deterministic lexical probability is combined with the semantic probability by noisy-OR. The encoder and matcher remain frozen during outcome training; concept annotations are used only for held-out grounding evaluation.


In [ ]:
from __future__ import annotations

import json
import math
import os
import random
import re
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
from from_n3c import *
import ast

CONCEPT_CSV = "../AdaptivePooling_MLHC/df_icd10_with_synonyms.csv"
TRAIN_JSON = "../AdaptivePooling_MLHC/ds_train_chest_trauma_ner.json"
DEV_JSON = "../AdaptivePooling_MLHC/ds_dev_chest_trauma_ner.json"
VAL_JSON = "../AdaptivePooling_MLHC/ds_test_chest_trauma_ner.json"
ALIAS_TABLE_OUT = "../AdaptivePooling_MLHC/icd10_3digit_aliases_generated.csv"

# The raw alias plus one clinical context are averaged into each prototype.
CONTEXT_TEMPLATES = [
    "Diagnosis: {}.",
    "Assessment: {}.",
    "Imaging demonstrates {}.",
    "Findings are consistent with {}.",
]

# Optional task-relevant additions; CSV synonyms remain the primary alias source.
CURATED_ALIASES = {
    "S22": ["rib fracture", "rib fractures", "broken ribs", "rib fx", "sternal fracture", "thoracic spine fracture"],
    "J93": ["pneumothorax", "collapsed lung", "PTX"],
    "J94": ["hemothorax", "haemothorax"],
    "S27": ["lung injury", "pulmonary injury", "intrathoracic injury"],
    "S42": ["clavicle fracture", "scapula fracture", "shoulder fracture"],
    "I95": ["hypotension", "low blood pressure"],
    "I46": ["cardiac arrest", "cardiopulmonary arrest"],
    "J96": ["respiratory failure", "acute respiratory failure", "resp failure"],
    "K72": ["hepatic failure", "liver failure"],
    "I50": ["heart failure", "congestive heart failure", "CHF"],
    "A41": ["sepsis", "septicemia"],
    "N17": ["acute kidney injury", "acute kidney failure", "AKI"],
    "R57": ["shock", "circulatory shock"],
}


def normalize_code(code: Any) -> str:
    return re.sub(r"[^A-Z0-9]", "", str(code).upper())[:3]


def normalize_alias(text: Any) -> str:
    return re.sub(r"\s+", " ", str(text)).strip(" ,;.")


def parse_synonyms(value: Any) -> List[str]:
    """Parse serialized Python/JSON lists without splitting commas inside terms."""
    if value is None:
        return []
    if not isinstance(value, (list, tuple, set, dict)):
        try:
            if pd.isna(value):
                return []
        except (TypeError, ValueError):
            pass

    parsed = value
    for _ in range(2):
        if not isinstance(parsed, str):
            break
        text = parsed.strip()
        if not text:
            return []
        new_value = None
        for candidate in (text, text.strip('"'), text.strip("'")):
            try:
                new_value = ast.literal_eval(candidate)
                break
            except (ValueError, SyntaxError):
                try:
                    new_value = json.loads(candidate)
                    break
                except (ValueError, TypeError, json.JSONDecodeError):
                    continue
        if new_value is None:
            parsed = re.split(r"[|;]", text)
            break
        if new_value == parsed:
            break
        parsed = new_value

    if isinstance(parsed, dict):
        parsed = list(parsed.values())
    elif not isinstance(parsed, (list, tuple, set)):
        parsed = [parsed]

    out = []
    for item in parsed:
        item = normalize_alias(item)
        if item and item.casefold() not in {"nan", "none", "null"}:
            out.append(item)
    return out


def make_aliases(code: str, name: str, synonyms: Any) -> List[str]:
    # Put informative synonyms/curated phrases before redundant name variants,
    # because only the first MAX_ALIASES are encoded.
    candidates = [
        name,
        *parse_synonyms(synonyms),
        *CURATED_ALIASES.get(code, []),
        name.replace("(s)", "s"),
        re.sub(r"\[([^]]+)\]", r"\1", name),
    ]

    aliases, seen = [], set()
    for value in candidates:
        value = normalize_alias(value)
        key = value.casefold()
        if value and key not in seen:
            seen.add(key)
            aliases.append(value)
    return aliases


def concept_group(code: str) -> str:
    try:
        return str(icd10_text(code))
    except Exception:
        try:
            return str(infer_chapter_from_code(code))
        except Exception:
            return code[:1]


df_concepts = pd.read_csv(CONCEPT_CSV)
required_columns = {"code", "name", "synonyms"}
missing_columns = required_columns - set(df_concepts.columns)
if missing_columns:
    raise ValueError(f"Missing required concept columns: {sorted(missing_columns)}")

concepts, seen_codes = [], set()
for _, row in df_concepts.iterrows():
    code = normalize_code(row["code"])
    if not code or code in seen_codes:
        continue
    seen_codes.add(code)
    name = normalize_alias(row["name"])
    if not name:
        continue
    concepts.append({
        "id": code,
        "text": name,
        "aliases": make_aliases(code, name, row["synonyms"]),
        "group": concept_group(code),
    })

# Remove aliases shared by multiple codes, while retaining each canonical name.
alias_to_codes = defaultdict(set)
for concept in concepts:
    for alias in concept["aliases"]:
        alias_to_codes[alias.casefold()].add(concept["id"])
for concept in concepts:
    canonical = concept["text"].casefold()
    concept["aliases"] = [
        alias for alias in concept["aliases"]
        if alias.casefold() == canonical or len(alias_to_codes[alias.casefold()]) == 1
    ]

pd.DataFrame([
    {
        "code": concept["id"],
        "name": concept["text"],
        "synonyms": repr(concept["aliases"][1:]),
        "aliases_used": "|".join(concept["aliases"]),
    }
    for concept in concepts
]).to_csv(ALIAS_TABLE_OUT, index=False)

with open(TRAIN_JSON) as f:
    train_samples = json.load(f)
with open(DEV_JSON) as f:
    dev_samples = json.load(f)
with open(VAL_JSON) as f:
    val_samples = json.load(f)
for samples in (train_samples, dev_samples, val_samples):
    for sample in samples:
        sample["label"] = int(sample["label"] >= 3)

print(f"Concepts: {len(concepts):,}; aliases: {sum(len(c['aliases']) for c in concepts):,}")
print("Alias-count summary:", pd.Series([len(c["aliases"]) for c in concepts]).describe().round(2).to_dict())
print("Outcome labels:", Counter(sample["label"] for sample in train_samples))


In [ ]:
class TextDataset(Dataset):
    def __init__(self, samples): self.samples = samples
    def __len__(self): return len(self.samples)
    def __getitem__(self, i): return self.samples[i]


def make_loader(samples, tokenizer, batch_size=4, max_length=512, shuffle=False, include_concepts=False):
    concept_to_idx = {c["id"]: i for i, c in enumerate(concepts)}

    def annotation_code(annotation):
        if isinstance(annotation, dict): raw = annotation.get("code", annotation.get("id"))
        elif isinstance(annotation, (list, tuple)) and len(annotation) >= 2: raw = annotation[1]
        else: raw = None
        return normalize_code(raw) if raw is not None else ""

    def collate(batch):
        encoded = tokenizer(
            [s["txt"] for s in batch], padding=True, truncation=True,
            max_length=max_length, return_tensors="pt",
        )
        encoded["labels"] = torch.tensor([s["label"] for s in batch], dtype=torch.long)
        if include_concepts:  # evaluation only
            y = torch.zeros(len(batch), len(concepts), dtype=torch.bool)
            for i, sample in enumerate(batch):
                for annotation in sample.get("concepts", []):
                    j = concept_to_idx.get(annotation_code(annotation))
                    if j is not None: y[i, j] = True
            encoded["concept_labels"] = y
        return encoded

    return DataLoader(
        TextDataset(samples), batch_size=batch_size, shuffle=shuffle,
        num_workers=0, pin_memory=True, collate_fn=collate,
    )

In [ ]:
STOP_TOKENS = {
    "of", "and", "or", "the", "a", "an", "in", "on", "with", "without",
    "other", "specified", "unspecified", "elsewhere", "classified", "not",
}


def informative_token(token: str) -> bool:
    token = token.replace("##", "").strip().lower()
    token = re.sub(r"[^a-z0-9]+", "", token)
    return bool(token) and token not in STOP_TOKENS


def find_subsequence(sequence: List[int], pattern: List[int]) -> Optional[int]:
    if not pattern or len(pattern) > len(sequence):
        return None
    n = len(pattern)
    for start in range(len(sequence) - n + 1):
        if sequence[start:start + n] == pattern:
            return start
    return None


class ExactAliasMatcher:
    """
    Fixed ontology lookup over unique tokenized aliases.

    It does not use note-level concept labels. Multi-token exact matches receive
    `exact_probability`; conservative one-token matches receive
    `single_token_probability`. Results are cached by tokenized note.
    """
    def __init__(
        self,
        tokenizer,
        concepts,
        max_aliases=8,
        max_alias_tokens=24,
        exact_probability=0.999,
        single_token_probability=0.995,
    ):
        self.num_concepts = len(concepts)
        self.max_alias_tokens = int(max_alias_tokens)
        self.exact_probability = float(exact_probability)
        self.single_token_probability = float(single_token_probability)
        self.nodes: List[Dict[int, int]] = [{}]
        self.terminals: List[List[Tuple[int, float]]] = [[]]
        self.cache: Dict[Tuple[int, ...], List[Tuple[int, float]]] = {}

        # Keep only aliases whose token sequence maps to exactly one ICD concept.
        pattern_to_concepts: Dict[Tuple[int, ...], set] = defaultdict(set)
        pattern_to_probability: Dict[Tuple[int, ...], float] = {}

        for concept_idx, concept in enumerate(concepts):
            for alias in concept["aliases"][:max_aliases]:
                alias = normalize_alias(alias)
                if not alias:
                    continue

                token_ids = tuple(
                    tokenizer(alias, add_special_tokens=False)["input_ids"]
                )
                if not token_ids or len(token_ids) > self.max_alias_tokens:
                    continue

                # Avoid very short generic one-word strings; retain ordinary
                # clinical terms and uppercase abbreviations such as AKI/PTX.
                compact = re.sub(r"[^A-Za-z0-9]", "", alias)
                surface_terms = re.findall(r"[A-Za-z0-9]+", alias)
                is_abbreviation = (
                    compact.isupper()
                    and 2 <= len(compact) <= 8
                    and compact.casefold() not in {"nos", "nec"}
                )
                if (
                    len(surface_terms) == 1
                    and len(compact) < 5
                    and not is_abbreviation
                ):
                    continue

                probability = (
                    self.exact_probability
                    if len(surface_terms) >= 2
                    else self.single_token_probability
                )
                pattern_to_concepts[token_ids].add(concept_idx)
                pattern_to_probability[token_ids] = max(
                    pattern_to_probability.get(token_ids, 0.0),
                    probability,
                )

        unique_patterns = 0
        covered_concepts = set()

        for pattern, concept_indices in pattern_to_concepts.items():
            if len(concept_indices) != 1:
                continue

            concept_idx = next(iter(concept_indices))
            self._add_pattern(
                pattern,
                concept_idx,
                pattern_to_probability[pattern],
            )
            unique_patterns += 1
            covered_concepts.add(concept_idx)

        self.n_unique_patterns = unique_patterns
        self.n_covered_concepts = len(covered_concepts)

    def _add_pattern(
        self,
        pattern: Tuple[int, ...],
        concept_idx: int,
        probability: float,
    ) -> None:
        node_idx = 0
        for token_id in pattern:
            child_idx = self.nodes[node_idx].get(token_id)
            if child_idx is None:
                child_idx = len(self.nodes)
                self.nodes[node_idx][token_id] = child_idx
                self.nodes.append({})
                self.terminals.append([])
            node_idx = child_idx
        self.terminals[node_idx].append((concept_idx, probability))

    def _match_one(self, token_ids: Tuple[int, ...]) -> List[Tuple[int, float]]:
        cached = self.cache.get(token_ids)
        if cached is not None:
            return cached

        matches: Dict[int, float] = {}
        n_tokens = len(token_ids)

        for start in range(n_tokens):
            node_idx = 0
            stop = min(n_tokens, start + self.max_alias_tokens)

            for position in range(start, stop):
                child_idx = self.nodes[node_idx].get(token_ids[position])
                if child_idx is None:
                    break

                node_idx = child_idx
                for concept_idx, probability in self.terminals[node_idx]:
                    matches[concept_idx] = max(
                        matches.get(concept_idx, 0.0),
                        probability,
                    )

        result = list(matches.items())
        self.cache[token_ids] = result
        return result

    @torch.no_grad()
    def __call__(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        device: torch.device,
        dtype: torch.dtype,
    ) -> torch.Tensor:
        input_ids_cpu = input_ids.detach().cpu()
        attention_cpu = attention_mask.detach().cpu().bool()
        batch_size = input_ids_cpu.shape[0]

        exact_probability = torch.zeros(
            batch_size,
            self.num_concepts,
            dtype=torch.float32,
        )

        for row_idx in range(batch_size):
            valid_ids = tuple(
                input_ids_cpu[row_idx, attention_cpu[row_idx]].tolist()
            )
            for concept_idx, probability in self._match_one(valid_ids):
                exact_probability[row_idx, concept_idx] = probability

        return exact_probability.to(device=device, dtype=dtype)

    def summary(self) -> Dict[str, int]:
        return {
            "unique_alias_patterns": self.n_unique_patterns,
            "concepts_with_exact_alias_pattern": self.n_covered_concepts,
            "cache_size": len(self.cache),
        }


@torch.no_grad()
def build_cls_embeddings(texts, tokenizer, encoder, device, batch_size=32, max_length=64):
    rows = []
    encoder.eval()
    for start in range(0, len(texts), batch_size):
        tok = tokenizer(
            texts[start:start + batch_size], padding=True, truncation=True,
            max_length=max_length, return_tensors="pt",
        ).to(device)
        rows.append(encoder(**tok).last_hidden_state[:, 0].cpu())
    return torch.cat(rows)


@torch.no_grad()
def build_prototype_bank(
    concepts, tokenizer, encoder, device,
    max_aliases=8, max_tokens=16, batch_size=32,
    context_templates=CONTEXT_TEMPLATES,
):
    """
    Build (C,A,T,H) prototypes. For each alias, average token embeddings from:
      1) the alias alone; and
      2) one rotating clinical context.
    This improves contextual robustness without increasing prototype-bank size.
    """
    metadata, texts = [], []
    for concept_idx, concept in enumerate(concepts):
        aliases = concept["aliases"][:max_aliases] or [concept["text"]]
        for alias_idx, alias in enumerate(aliases):
            alias_ids = tokenizer(alias, add_special_tokens=False)["input_ids"]
            alias_tokens = tokenizer.convert_ids_to_tokens(alias_ids)
            keep = [i for i, token in enumerate(alias_tokens) if informative_token(token)]
            if not keep:
                keep = list(range(len(alias_ids)))
            keep = keep[:max_tokens]

            templates = ["{}"]
            if context_templates:
                templates.append(context_templates[alias_idx % len(context_templates)])
            for template in templates:
                texts.append(template.format(alias))
                metadata.append((concept_idx, alias_idx, alias_ids, keep))

    C, H = len(concepts), encoder.config.hidden_size
    dtype = torch.float16 if device.type == "cuda" else torch.float32
    bank = torch.zeros(C, max_aliases, max_tokens, H, dtype=dtype)
    counts = torch.zeros(C, max_aliases, max_tokens, dtype=torch.float32)
    encoder.eval()

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]
        tok = tokenizer(
            batch_texts, padding=True, truncation=True,
            max_length=max(64, max_tokens + 24), return_tensors="pt",
        ).to(device)
        hidden = encoder(**tok).last_hidden_state

        for row_idx, (concept_idx, alias_idx, alias_ids, keep) in enumerate(
            metadata[start:start + batch_size]
        ):
            valid_length = int(tok["attention_mask"][row_idx].sum().item())
            valid_ids = tok["input_ids"][row_idx, :valid_length].tolist()
            alias_start = find_subsequence(valid_ids, alias_ids)
            if alias_start is None:
                continue
            for output_pos, alias_token_pos in enumerate(keep):
                note_pos = alias_start + alias_token_pos
                if note_pos >= hidden.shape[1]:
                    continue
                bank[concept_idx, alias_idx, output_pos] += (
                    hidden[row_idx, note_pos].detach().cpu().to(dtype)
                )
                counts[concept_idx, alias_idx, output_pos] += 1.0

    valid = counts > 0
    bank = bank / counts.clamp(min=1.0).unsqueeze(-1).to(bank.dtype)
    weights = valid.float()
    return bank, weights


@dataclass
class AVOOutput:
    logits: torch.Tensor
    token_logits: torch.Tensor
    A: torch.Tensor
    V: torch.Tensor
    O: torch.Tensor
    sim: torch.Tensor
    A_pool: torch.Tensor
    AV_pool: torch.Tensor
    concept_logits: torch.Tensor
    semantic_concept_logits: torch.Tensor
    exact_match_probability: torch.Tensor


class LateInteractionAVOHead(nn.Module):
    """Local phrase coverage using fixed token-level ICD aliases."""
    def __init__(
        self, concept_emb, prototype_bank, prototype_weights,
        dv=256, num_outputs=2, match_margin=0.65,
        match_temperature=0.05, concept_chunk_size=32,
        local_window_size=17,
    ):
        super().__init__()
        if local_window_size < 1 or local_window_size % 2 == 0:
            raise ValueError("local_window_size must be a positive odd integer.")
        C, H = concept_emb.shape
        self.C, self.H, self.dv = C, H, dv
        self.match_margin = float(match_margin)
        self.match_temperature = float(match_temperature)
        self.concept_chunk_size = int(concept_chunk_size)
        self.local_window_size = int(local_window_size)
        self.register_buffer("concept_emb", concept_emb.detach().clone())
        self.register_buffer("prototype_bank", prototype_bank.detach().clone())
        self.register_buffer("prototype_weights", prototype_weights.detach().clone())

        # Shared frozen identity map preserves SapBERT geometry and exact matches.
        self.Wmatch = nn.Linear(H, H, bias=False)
        nn.init.eye_(self.Wmatch.weight)
        for parameter in self.Wmatch.parameters():
            parameter.requires_grad = False

        self.Wv = nn.Linear(H, dv, bias=False)
        self.O = nn.Parameter(torch.randn(dv, num_outputs) * 0.02)
        self.bias = nn.Parameter(torch.zeros(num_outputs))

    def forward(self, token_embs, token_mask, exact_match_probability=None):
        q = F.normalize(self.Wmatch(token_embs), dim=-1)
        B, L, _ = q.shape
        A_real = q.new_empty(B, L, self.C)
        raw_presence = q.new_empty(B, self.C)

        for start in range(0, self.C, self.concept_chunk_size):
            end = min(start + self.concept_chunk_size, self.C)
            proto = self.prototype_bank[start:end].to(device=q.device, dtype=q.dtype)
            weight = self.prototype_weights[start:end].to(q.device)
            k = F.normalize(self.Wmatch(proto), dim=-1)

            # sim: (B, note_token, concept, alias, alias_token)
            sim = torch.einsum("blh,cath->blcat", q, k)
            valid_proto = weight > 0
            sim = sim.masked_fill(~token_mask[:, :, None, None, None], -1e4)
            sim = sim.masked_fill(~valid_proto[None, None], -1e4)

            # Token evidence used only for token-level visualization.
            token_support = sim.amax(dim=(-1, -2))
            A_real[:, :, start:end] = torch.sigmoid(
                (token_support - self.match_margin) / self.match_temperature
            ).masked_fill(~token_mask.unsqueeze(-1), 0.0)

            # Require the alias tokens to be covered within one local note window.
            # Pool along note-token dimension for every (concept, alias, alias-token).
            pooled = sim.permute(0, 2, 3, 4, 1).reshape(-1, 1, L)
            pooled = F.max_pool1d(
                pooled,
                kernel_size=self.local_window_size,
                stride=1,
                padding=self.local_window_size // 2,
            )
            pooled = pooled.reshape(B, end - start, *weight.shape[1:], L)
            pooled = pooled.permute(0, 1, 2, 4, 3)  # (B,c,A,L,T)

            denom = weight.sum(dim=-1).clamp(min=1.0)
            alias_window_score = (
                pooled * weight[None, :, :, None, :]
            ).sum(dim=-1) / denom[None, :, :, None]
            alias_window_score = alias_window_score.masked_fill(
                ~(valid_proto.any(dim=-1))[None, :, :, None], -1e4
            )
            raw_presence[:, start:end] = alias_window_score.amax(dim=(-1, -2))

        semantic_concept_logits = (
            raw_presence - self.match_margin
        ) / self.match_temperature
        semantic_presence = torch.sigmoid(semantic_concept_logits)

        # Fixed exact-alias evidence is combined with semantic evidence by
        # noisy-OR. This guarantees high presence for a unique verbatim alias
        # while retaining semantic scores to order multiple exact matches.
        if exact_match_probability is None:
            exact_match_probability = torch.zeros_like(semantic_presence)
        else:
            exact_match_probability = exact_match_probability.to(
                device=semantic_presence.device,
                dtype=semantic_presence.dtype,
            ).clamp(0.0, 1.0 - 1e-6)

        presence = 1.0 - (
            (1.0 - semantic_presence)
            * (1.0 - exact_match_probability)
        )
        presence = presence.clamp(1e-6, 1.0 - 1e-6)
        concept_logits = torch.logit(presence)

        # Keep a zero-contribution NULL row for compatibility with the original AVO layer.
        A_null = 1.0 - A_real.max(dim=-1, keepdim=True).values
        A = torch.cat([A_null, A_real], dim=-1)
        pool_null = 1.0 - presence.max(dim=-1, keepdim=True).values
        A_pool = torch.cat([pool_null, presence], dim=-1)

        V_real = self.Wv(self.concept_emb)
        V = torch.cat([V_real.new_zeros(1, self.dv), V_real], dim=0)
        AV_pool = A_pool @ V
        logits = AV_pool @ self.O + self.bias

        token_logits = torch.logit(A.clamp(1e-6, 1 - 1e-6))
        return AVOOutput(
            logits=logits,
            token_logits=token_logits,
            A=A,
            V=V,
            O=self.O,
            sim=A,
            A_pool=A_pool,
            AV_pool=AV_pool,
            concept_logits=concept_logits,
            semantic_concept_logits=semantic_concept_logits,
            exact_match_probability=exact_match_probability,
        )


class LateInteractionAVOModel(nn.Module):
    def __init__(self, encoder, head, tokenizer, exact_alias_matcher=None):
        super().__init__()
        self.text_encoder, self.head = encoder, head
        self.special_ids = tokenizer.all_special_ids
        self.exact_alias_matcher = exact_alias_matcher
        for parameter in self.text_encoder.parameters():
            parameter.requires_grad = False

    def train(self, mode=True):
        super().train(mode)
        self.text_encoder.eval()
        return self

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids

        token_embs = self.text_encoder(**kwargs).last_hidden_state
        token_mask = attention_mask.bool()
        for token_id in self.special_ids:
            token_mask &= input_ids != token_id

        exact_match_probability = None
        if self.exact_alias_matcher is not None:
            exact_match_probability = self.exact_alias_matcher(
                input_ids=input_ids,
                attention_mask=attention_mask,
                device=token_embs.device,
                dtype=token_embs.dtype,
            )

        return self.head(
            token_embs,
            token_mask,
            exact_match_probability=exact_match_probability,
        ), token_mask


In [ ]:
def group_lasso(beta, groups, lambda_=1e-3):
    penalty = beta.new_zeros(())
    for group in groups:
        idx = torch.as_tensor(group, device=beta.device, dtype=torch.long) + 1
        penalty = penalty + math.sqrt(len(group)) * beta.index_select(0, idx).norm()
    return lambda_ * penalty


def build_groups(concepts):
    by_group = defaultdict(list)
    for i, concept in enumerate(concepts):
        by_group[concept["group"]].append(i)
    return list(by_group.values())


@torch.no_grad()
def evaluate_outcome(model, loader, device):
    model.eval(); ys, ps = [], []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items() if isinstance(v, torch.Tensor)}
        out, _ = model(
            batch["input_ids"], batch["attention_mask"], batch.get("token_type_ids")
        )
        ys.append(batch["labels"].cpu().numpy())
        ps.append(torch.softmax(out.logits, -1)[:, 1].cpu().numpy())
    y, p = np.concatenate(ys), np.concatenate(ps)
    return {"AUROC": roc_auc_score(y, p), "AUPR": average_precision_score(y, p)}


@torch.no_grad()
def evaluate_grounding(
    model, loader, device, ks=(1, 5, 10),
    contribution=False, semantic_only=False,
):
    model.eval(); hits = {k: 0 for k in ks}; n = 0
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items() if isinstance(v, torch.Tensor)}
        out, _ = model(
            batch["input_ids"], batch["attention_mask"], batch.get("token_type_ids")
        )
        if contribution:
            beta = (out.V @ out.O)[1:]
            scores = out.A_pool[:, 1:] * (
                beta[:, 1] - beta[:, 0]
            ).unsqueeze(0)
        elif semantic_only:
            scores = out.semantic_concept_logits
        else:
            scores = out.concept_logits  # semantic + fixed exact-alias evidence

        labels = batch["concept_labels"].bool()
        valid = labels.any(dim=1)
        if not valid.any():
            continue
        top = scores[valid].topk(min(max(ks), scores.shape[1]), dim=1).indices
        labels = labels[valid]; n += int(valid.sum())
        for k in ks:
            kk = min(k, top.shape[1])
            hits[k] += int(labels.gather(1, top[:, :kk]).any(dim=1).sum())
    if contribution:
        prefix = "contributor"
    elif semantic_only:
        prefix = "semantic_presence"
    else:
        prefix = "presence"
    return {
        "n_labeled_notes": n,
        **{
            f"{prefix}_hit@{k}": hits[k] / max(n, 1)
            for k in ks
        },
    }


@torch.no_grad()
def exact_match_test(
    model, tokenizer, concepts, device,
    max_examples=512, batch_size=8, contextual=False,
    semantic_only=False,
):
    """Feed stored aliases either alone or in a short clinical context."""
    examples = [(alias, j, a) for j, c in enumerate(concepts) for a, alias in enumerate(c["aliases"][:2])]
    rng = random.Random(SEED); rng.shuffle(examples)
    if max_examples is not None:
        examples = examples[:max_examples]

    ranks, probabilities = [], []
    model.eval()
    for start in range(0, len(examples), batch_size):
        chunk = examples[start:start + batch_size]
        texts = []
        for alias, _, alias_idx in chunk:
            if contextual:
                template = CONTEXT_TEMPLATES[alias_idx % len(CONTEXT_TEMPLATES)]
                texts.append(template.format(alias))
            else:
                texts.append(alias)
        tok = tokenizer(
            texts, padding=True, truncation=True,
            max_length=64, return_tensors="pt",
        ).to(device)
        out, _ = model(**tok)
        scores = (
            out.semantic_concept_logits
            if semantic_only
            else out.concept_logits
        )
        target = torch.tensor([x[1] for x in chunk], device=device)
        target_logit = scores.gather(1, target[:, None]).squeeze(1)
        rank = 1 + (scores > target_logit[:, None]).sum(dim=1)
        ranks.extend(rank.cpu().tolist())
        probabilities.extend(torch.sigmoid(target_logit).cpu().tolist())

    ranks = np.asarray(ranks); probabilities = np.asarray(probabilities)
    return {
        "n": len(ranks),
        "hit@1": float((ranks <= 1).mean()),
        "hit@5": float((ranks <= 5).mean()),
        "hit@10": float((ranks <= 10).mean()),
        "median_target_probability": float(np.median(probabilities)),
    }



@torch.no_grad()
def evaluate_exact_alias_boost(model, loader, device, ks=(1, 5, 10)):
    """Compare semantic-only and exact-boosted ranking on exact-matched labels."""
    model.eval()
    max_k = max(ks)
    n_exact_pairs = 0
    semantic_captured = {k: 0 for k in ks}
    boosted_captured = {k: 0 for k in ks}
    semantic_probabilities = []
    boosted_probabilities = []

    for batch in loader:
        batch = {
            key: value.to(device)
            for key, value in batch.items()
            if isinstance(value, torch.Tensor)
        }
        out, _ = model(
            batch["input_ids"],
            batch["attention_mask"],
            batch.get("token_type_ids"),
        )

        labels = batch["concept_labels"].bool()
        exact = out.exact_match_probability > 0
        exact_labeled = labels & exact
        if not exact_labeled.any():
            continue

        semantic_scores = out.semantic_concept_logits
        boosted_scores = out.concept_logits
        semantic_top = semantic_scores.topk(max_k, dim=1).indices
        boosted_top = boosted_scores.topk(max_k, dim=1).indices

        pair_rows, pair_cols = torch.where(exact_labeled)
        n_exact_pairs += pair_rows.numel()
        semantic_probabilities.extend(
            torch.sigmoid(
                semantic_scores[pair_rows, pair_cols]
            ).cpu().tolist()
        )
        boosted_probabilities.extend(
            torch.sigmoid(
                boosted_scores[pair_rows, pair_cols]
            ).cpu().tolist()
        )

        for k in ks:
            semantic_captured[k] += int(
                (
                    semantic_top[pair_rows, :k]
                    == pair_cols.unsqueeze(1)
                ).any(dim=1).sum()
            )
            boosted_captured[k] += int(
                (
                    boosted_top[pair_rows, :k]
                    == pair_cols.unsqueeze(1)
                ).any(dim=1).sum()
            )

    return {
        "n_exact_visible_labeled_pairs": n_exact_pairs,
        "semantic_mean_probability":
            float(np.mean(semantic_probabilities))
            if semantic_probabilities else float("nan"),
        "boosted_mean_probability":
            float(np.mean(boosted_probabilities))
            if boosted_probabilities else float("nan"),
        **{
            f"semantic_exact_pair_recall@{k}":
                semantic_captured[k] / max(n_exact_pairs, 1)
            for k in ks
        },
        **{
            f"boosted_exact_pair_recall@{k}":
                boosted_captured[k] / max(n_exact_pairs, 1)
            for k in ks
        },
    }


def annotation_code(annotation):
    if isinstance(annotation, dict):
        raw = annotation.get("code", annotation.get("id"))
    elif isinstance(annotation, (list, tuple)) and len(annotation) >= 2:
        raw = annotation[1]
    else:
        raw = None
    return normalize_code(raw) if raw is not None else ""


def audit_alias_visibility(samples, tokenizer, concepts, max_length=512, max_aliases=8):
    """How often a labeled code has a stored alias visible in the truncated note."""
    alias_ids = {
        concept["id"]: [
            tokenizer(alias, add_special_tokens=False)["input_ids"]
            for alias in concept["aliases"][:max_aliases]
            if alias.strip()
        ]
        for concept in concepts
    }
    n_notes = n_visible_notes = n_pairs = n_visible_pairs = 0

    for sample in samples:
        codes = {annotation_code(x) for x in sample.get("concepts", [])}
        codes.discard("")
        if not codes:
            continue
        note_ids = tokenizer(
            sample["txt"], add_special_tokens=False,
            truncation=True, max_length=max_length,
        )["input_ids"]
        n_notes += 1
        any_visible = False
        for code in codes:
            n_pairs += 1
            visible = any(
                find_subsequence(note_ids, pattern) is not None
                for pattern in alias_ids.get(code, [])
            )
            n_visible_pairs += int(visible)
            any_visible |= visible
        n_visible_notes += int(any_visible)

    return {
        "n_labeled_notes": n_notes,
        "n_labeled_pairs": n_pairs,
        "visible_labeled_pair_fraction": n_visible_pairs / max(n_pairs, 1),
        "notes_with_visible_labeled_alias": n_visible_notes / max(n_notes, 1),
    }


In [ ]:
MODEL_NAME = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
BATCH_SIZE = 4
MAX_LENGTH = 512
DV = 256
OUTCOME_WARMUP_EPOCHS = 1
OUTCOME_EPOCHS = 2
OUTCOME_LR = 1e-5
MATCH_MARGIN = 0.65
MATCH_TEMPERATURE = 0.05
CONCEPT_CHUNK_SIZE = 32
MAX_ALIASES = 8
MAX_ALIAS_TOKENS = 16
LOCAL_WINDOW_SIZE = 17
EXACT_MATCH_EXAMPLES = 512  # set None to test all stored aliases

# Fixed ontology-only exact-alias branch; no note concept labels are used.
ENABLE_EXACT_ALIAS_BOOST = True
EXACT_ALIAS_PROBABILITY = 0.999
EXACT_SINGLE_TOKEN_PROBABILITY = 0.995
MAX_EXACT_ALIAS_TOKENS = 24


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)

exact_alias_matcher = None
if ENABLE_EXACT_ALIAS_BOOST:
    exact_alias_matcher = ExactAliasMatcher(
        tokenizer=tokenizer,
        concepts=concepts,
        max_aliases=MAX_ALIASES,
        max_alias_tokens=MAX_EXACT_ALIAS_TOKENS,
        exact_probability=EXACT_ALIAS_PROBABILITY,
        single_token_probability=EXACT_SINGLE_TOKEN_PROBABILITY,
    )
    print("Exact-alias matcher:", exact_alias_matcher.summary())
train_loader = make_loader(train_samples, tokenizer, BATCH_SIZE, MAX_LENGTH, shuffle=True)
dev_loader = make_loader(dev_samples, tokenizer, BATCH_SIZE, MAX_LENGTH)
val_loader = make_loader(val_samples, tokenizer, BATCH_SIZE, MAX_LENGTH)
dev_grounding_loader = make_loader(dev_samples, tokenizer, BATCH_SIZE, MAX_LENGTH, include_concepts=True)
val_grounding_loader = make_loader(val_samples, tokenizer, BATCH_SIZE, MAX_LENGTH, include_concepts=True)


class BlackBoxLM(nn.Module):
    def __init__(self, encoder):
        super().__init__(); self.encoder = encoder
        self.head = nn.Linear(encoder.config.hidden_size, 2)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids
        hidden = self.encoder(**kwargs).last_hidden_state
        return self.head(hidden[:, 0])


blackbox = BlackBoxLM(encoder).to(DEVICE)
optimizer = torch.optim.AdamW(blackbox.parameters(), lr=1e-5)
for epoch in range(OUTCOME_WARMUP_EPOCHS):
    blackbox.train(); total = 0.0
    for batch in train_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items() if isinstance(v, torch.Tensor)}
        logits = blackbox(batch["input_ids"], batch["attention_mask"], batch.get("token_type_ids"))
        loss = F.cross_entropy(logits, batch["labels"])
        optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
        total += loss.item()
    print(f"Outcome warm-up {epoch + 1}: loss={total / len(train_loader):.4f}")


In [ ]:
# Build the fixed ontology matcher after outcome warm-up.
encoder = blackbox.encoder
concept_emb = build_cls_embeddings(
    [concept["text"] for concept in concepts], tokenizer, encoder, DEVICE,
    batch_size=32, max_length=64,
).to(DEVICE)
prototype_bank, prototype_weights = build_prototype_bank(
    concepts, tokenizer, encoder, DEVICE,
    max_aliases=MAX_ALIASES,
    max_tokens=MAX_ALIAS_TOKENS,
    batch_size=32,
    context_templates=CONTEXT_TEMPLATES,
)

head = LateInteractionAVOHead(
    concept_emb=concept_emb,
    prototype_bank=prototype_bank,
    prototype_weights=prototype_weights,
    dv=DV,
    num_outputs=2,
    match_margin=MATCH_MARGIN,
    match_temperature=MATCH_TEMPERATURE,
    concept_chunk_size=CONCEPT_CHUNK_SIZE,
    local_window_size=LOCAL_WINDOW_SIZE,
).to(DEVICE)
model = LateInteractionAVOModel(
    encoder,
    head,
    tokenizer,
    exact_alias_matcher=exact_alias_matcher,
).to(DEVICE)
del concept_emb, prototype_bank, prototype_weights
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Semantic exact-alias diagnostic:", exact_match_test(
    model, tokenizer, concepts, DEVICE,
    max_examples=EXACT_MATCH_EXAMPLES,
    contextual=False,
    semantic_only=True,
))
print("Exact-boosted alias diagnostic:", exact_match_test(
    model, tokenizer, concepts, DEVICE,
    max_examples=EXACT_MATCH_EXAMPLES,
    contextual=False,
    semantic_only=False,
))
print("Contextual exact-boosted diagnostic:", exact_match_test(
    model, tokenizer, concepts, DEVICE,
    max_examples=EXACT_MATCH_EXAMPLES,
    contextual=True,
    semantic_only=False,
))
print("Dev alias visibility:", audit_alias_visibility(
    dev_samples, tokenizer, concepts,
    max_length=MAX_LENGTH, max_aliases=MAX_ALIASES,
))
print("Dev semantic-only grounding before outcome training:",
      evaluate_grounding(
          model, dev_grounding_loader, DEVICE,
          semantic_only=True,
      ))
print("Dev grounding with exact-alias boost before outcome training:",
      evaluate_grounding(model, dev_grounding_loader, DEVICE))
print("Dev exact-alias boost effect:",
      evaluate_exact_alias_boost(
          model, dev_grounding_loader, DEVICE,
      ))


In [ ]:
groups = build_groups(concepts)

# Explicitly freeze the encoder and matcher. Only the outcome layer is trained.
for parameter in model.parameters():
    parameter.requires_grad = False
for parameter in model.head.Wv.parameters():
    parameter.requires_grad = True
model.head.O.requires_grad_(True)
model.head.bias.requires_grad_(True)

optimizer = torch.optim.AdamW(
    [*model.head.Wv.parameters(), model.head.O, model.head.bias],
    lr=OUTCOME_LR,
)

# Fixed batch verifies that outcome training never changes concept presence.
fixed_batch = next(iter(dev_grounding_loader))
fixed_batch = {
    key: value.to(DEVICE)
    for key, value in fixed_batch.items()
    if isinstance(value, torch.Tensor)
}


@torch.no_grad()
def fixed_presence_logits():
    model.eval()
    out, _ = model(
        fixed_batch["input_ids"],
        fixed_batch["attention_mask"],
        fixed_batch.get("token_type_ids"),
    )
    return out.concept_logits.cpu()


presence_reference = fixed_presence_logits()

for epoch in range(1, OUTCOME_EPOCHS + 1):
    model.train(); total = 0.0
    for batch in train_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items() if isinstance(v, torch.Tensor)}
        out, _ = model(
            batch["input_ids"], batch["attention_mask"], batch.get("token_type_ids")
        )
        beta = out.V @ out.O
        loss = F.cross_entropy(out.logits, batch["labels"]) + group_lasso(beta, groups, 1e-3)
        optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
        total += loss.item()

    max_presence_change = (
        fixed_presence_logits() - presence_reference
    ).abs().max().item()
    if max_presence_change > 1e-6:
        raise RuntimeError(f"Concept matcher changed during outcome training: {max_presence_change:g}")

    print(f"Epoch {epoch}: loss={total / len(train_loader):.4f}")
    print(f"  maximum presence change: {max_presence_change:.2e}")
    print("  outcome:", evaluate_outcome(model, dev_loader, DEVICE))
    print("  semantic presence:", evaluate_grounding(
        model, dev_grounding_loader, DEVICE, semantic_only=True
    ))
    print("  exact-boosted presence:", evaluate_grounding(
        model, dev_grounding_loader, DEVICE
    ))
    print("  contributor:", evaluate_grounding(
        model, dev_grounding_loader, DEVICE, contribution=True
    ))

print("Final validation/test outcome:", evaluate_outcome(model, val_loader, DEVICE))
print("Final validation/test semantic-only grounding:",
      evaluate_grounding(
          model, val_grounding_loader, DEVICE,
          semantic_only=True,
      ))
print("Final validation/test exact-boosted grounding:",
      evaluate_grounding(model, val_grounding_loader, DEVICE))
print("Final validation/test exact-alias boost effect:",
      evaluate_exact_alias_boost(
          model, val_grounding_loader, DEVICE,
      ))
print("Final validation/test contributor grounding:",
      evaluate_grounding(model, val_grounding_loader, DEVICE, contribution=True))
